In [8]:
import sys
import math
import pandas as pd
sys.path.append("..")
sys.path.append("../player_score")

from rosters_2026 import rosters_2026
from steps.scoring import CLUB_TIERS

CLUB_TIER_WEIGHTS = {
    "Tier_1": 1.00,
    "Tier_2": 0.80,
    "Tier_3": 0.60,
    "Tier_4": 0.30,
    "Tier_5": 0.10,
    "__default__": 0.20,
}

def get_club_tier_weight(club: str) -> float:
    for tier, clubs in CLUB_TIERS.items():
        if club in clubs:
            return CLUB_TIER_WEIGHTS[tier]
    return CLUB_TIER_WEIGHTS["__default__"]

def compute_club_cohesion(country: str) -> dict:
    players = rosters_2026.get(country, {})
    if not players:
        return {"country": country, "top_club": None,
                "cohesion_count": 0, "cohesion_score": 0.0}

    clubs = [info["club"] for info in players.values()]
    club_counts = pd.Series(clubs).value_counts()
    
    top_club = club_counts.index[0]
    top_count = int(club_counts.iloc[0])
    club_weight = get_club_tier_weight(top_club)
    count_score = min(math.log(top_count + 1) / math.log(9) * 100, 100)
    cohesion_score = count_score * club_weight

    return {
        "country": country,
        "top_club": top_club,
        "cohesion_count": top_count,
        "club_tier_weight": club_weight,
        "cohesion_score": round(cohesion_score, 1),
    }

# Build table
rows = [compute_club_cohesion(c) for c in rosters_2026]
cohesion_df = pd.DataFrame(rows).sort_values("cohesion_score", ascending=False)
print(cohesion_df.to_string(index=False))

               country                       top_club  cohesion_count  club_tier_weight  cohesion_score
               Germany                  Bayern Munich               7               1.0            94.6
                 Spain                      Barcelona               6               1.0            88.6
                France            Paris Saint-Germain               5               1.0            81.5
               England                Manchester City               4               1.0            73.2
                Brazil                    Real Madrid               4               1.0            73.2
           Netherlands                      Liverpool               4               1.0            73.2
              Portugal            Paris Saint-Germain               4               1.0            73.2
                 Italy                    Inter Milan               4               1.0            73.2
             Argentina                Atlético Madrid           

In [2]:
# Now import
from player_score.player_score_pipeline import get_player_scores
scored = get_player_scores(verbose=True)

PLAYER SCORE PIPELINE

[1/8] Loading raw data...

Loading: recent_club_players
Seasons: ['2021/2022', '2022/2023', '2023/2024']

  ── 2021/2022 ──
    ✓ xg__player__totals.csv — 293 rows, 284 players
    ✓ progression__player__profile.csv — 1,175 rows, 1123 players
    ✓ advanced__player__packing.csv — 5,104 rows, 1053 players
    ✓ advanced__player__xg_chain.csv — 5,403 rows, 1151 players
    ✓ advanced__player__xg_buildup.csv — 4,731 rows, 1083 players
    ✓ advanced__player__network_centrality.csv — 6,194 rows, 1244 players
    ✓ defensive__player__profile.csv — 1,919 rows, 650 players
    ✓ defensive__player__pressures.csv — 5,647 rows, 1167 players

  ── 2022/2023 ──
    ✓ xg__player__totals.csv — 134 rows, 131 players
    ✓ progression__player__profile.csv — 884 rows, 846 players
    ✓ advanced__player__packing.csv — 2,417 rows, 874 players
    ✓ advanced__player__xg_chain.csv — 2,421 rows, 888 players
    ✓ advanced__player__xg_buildup.csv — 2,125 rows, 822 players
    ✓ advance

In [15]:
import importlib
import player_aggregator
importlib.reload(player_aggregator)
from player_aggregator import build_player_quality_table

player_quality_df = build_player_quality_table(scored)
print(player_quality_df.head(20).to_string(index=False))

Exported player quality table → /Users/5soomi/Desktop/school-project/soccer-analytics-capstone-template/composite_score/../player_score/outputs/player_quality_2026.csv
Exported player details → /Users/5soomi/Desktop/school-project/soccer-analytics-capstone-template/composite_score/../player_score/outputs/player_details_2026.csv (165 players)
       country  player_quality_score  effective_score  player_coverage_confidence  n_players_scored                         top_player  top_player_score
         Spain                 64.18            64.18                       1.000                12                   Fabián Ruiz Peña             79.90
        France                 62.95            62.95                       1.000                13               Kylian Mbappé Lottin             85.64
       Germany                 60.66            59.18                       0.909                10                      Florian Wirtz             66.35
     Argentina                 59.27        

In [4]:
# Cell 1 — imports
import sys
sys.path.insert(0, '..')
sys.path.insert(0, '../player_score')
sys.path.insert(0, '../player_score/steps')
# Cell 2 — get player scores
import importlib
import player_score.player_score_pipeline
import composite_score
importlib.reload(composite_score)
importlib.reload(player_score.player_score_pipeline)

from player_score.player_score_pipeline import get_player_scores
scored = get_player_scores(verbose=True)
# Cell 3 — get team readiness scores
from composite_scorer import get_team_readiness_scores
archetype_df = pd.read_csv("../tactical_clustering/outputs/team_archetypes.csv")
df = get_team_readiness_scores(scored, archetype_df=archetype_df, verbose=True)
print(df[["country", "final_score", "player_quality_score",
          "fifa_score", "club_cohesion_score",
          "coach_tenure_score", "tournament_exp_score"]].head(20).to_string())

PLAYER SCORE PIPELINE

[1/8] Loading raw data...

Loading: recent_club_players
Seasons: ['2021/2022', '2022/2023', '2023/2024']

  ── 2021/2022 ──
    ✓ xg__player__totals.csv — 293 rows, 284 players
    ✓ progression__player__profile.csv — 1,175 rows, 1123 players
    ✓ advanced__player__packing.csv — 5,104 rows, 1053 players
    ✓ advanced__player__xg_chain.csv — 5,403 rows, 1151 players
    ✓ advanced__player__xg_buildup.csv — 4,731 rows, 1083 players
    ✓ advanced__player__network_centrality.csv — 6,194 rows, 1244 players
    ✓ defensive__player__profile.csv — 1,919 rows, 650 players
    ✓ defensive__player__pressures.csv — 5,647 rows, 1167 players

  ── 2022/2023 ──
    ✓ xg__player__totals.csv — 134 rows, 131 players
    ✓ progression__player__profile.csv — 884 rows, 846 players
    ✓ advanced__player__packing.csv — 2,417 rows, 874 players
    ✓ advanced__player__xg_chain.csv — 2,421 rows, 888 players
    ✓ advanced__player__xg_buildup.csv — 2,125 rows, 822 players
    ✓ advance

In [5]:
GROUPS_2026 = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Côte d'Ivoire", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

In [14]:
import importlib
import monte_carlo
importlib.reload(monte_carlo)
from monte_carlo import run_monte_carlo

mc_df = run_monte_carlo(df, n_simulations=10000, verbose=True)
print(mc_df[["team", "group", "readiness_score", "p_champion",
             "p_runner_up", "p_semi_final", "p_quarter_final",
             "p_group_exit"]].to_string())

  ⚠ 'Ghana' not in readiness scores — using default 45.0

Running 10,000 simulations...
  Simulation 0/10,000...
  Simulation 1,000/10,000...
  Simulation 2,000/10,000...
  Simulation 3,000/10,000...
  Simulation 4,000/10,000...
  Simulation 5,000/10,000...
  Simulation 6,000/10,000...
  Simulation 7,000/10,000...
  Simulation 8,000/10,000...
  Simulation 9,000/10,000...

✓ Done — 10,000 simulations complete

TOP 10 CHAMPION PROBABILITIES:
               team group  readiness_score  p_champion  p_semi_final  p_quarter_final
rank                                                                                 
1             Spain     H            68.71         8.3           8.6             12.5
2         Argentina     J            68.73         8.1           7.5             11.6
3            France     I            66.10         6.9           7.2             12.2
4           Germany     E            63.16         6.5           7.8             13.4
5            Brazil     C            62.